# MC Truth Interface

`pyutils.pymcutils` contains accessors related to the MC truth information. It can be used in the same way as any other `pyutils` class.

First import the relevant python packages, followed by other `pyutils` classes that will aid the analysis:

In [1]:
# Import external packages
import awkward as ak

# Import the Procssor class
from pyutils.pyprocess import Processor 
from pyutils.pymcutil import MC

In [2]:
# Initialise the Importer with increased verbosity 
# verbosity=0 will only show errors
processor = Processor(verbosity=2)

# Define the path to our example file
file_name = "/exp/mu2e/data/users/sophie/ensembles/MDS1/MDS1av0.root"

# Define the branches we want to access
# For a complete list of available branches, see:
# https://github.com/Mu2e/EventNtuple/blob/main/doc/branches.md
# Also refer to ntuplehelper, available after mu2e setup EventNtuple
branches = ["trkmc", "trkmcsim"]

# Import the branches from the file
# This loads the data into memory and returns an awkward array
data = processor.process_data(
  file_name=file_name,
  branches=branches
)

mc_example = MC()
mc_example.count_particle_types(data)

[pyprocess] [INFO] Initialised Processor:
	path = 'EventNtuple/ntuple'
	use_remote = False
	verbosity=2


[pyprocess] [OK] Completed process on /exp/mu2e/data/users/sophie/ensembles/MDS1/MDS1av0.root
[pymcutil] [INFO] Initialised
[pyvector] [INFO] Initialised Vector with verbosity = 1
[pyvector] [OK] Created 3D 'pos' vector
[pyvector] [OK] Got 'pos' rho
[pymcutil] [INFO] ===== MC truth yields for full momentum and time range=====
[pymcutil] [INFO] N_DIO: 13085
[pymcutil] [INFO] N_IPA: 0
[pymcutil] [INFO] N_CEM: 106
[pymcutil] [INFO] N_CEP: 0
[pymcutil] [INFO] N_eRPC: 0
[pymcutil] [INFO] N_iRPC: 0
[pymcutil] [INFO] N_iRMC: 0
[pymcutil] [INFO] N_eRMC: 0
[pymcutil] [INFO] N_cosmic: 14885
[pymcutil] [INFO] N_others: 26093


[pymcutil] [INFO] returned particle count length 28076


<Array [-1, 166, -1, -1, -1, ..., -1, 166, -1, 166, -1] type='28076 * int64'>

## Selecting by MC truth

`MC` also provides masks with the same shape as `trkmcsim`. Combine them with `is_track_particle`, which selects the MC particle that contributed most to each track, to select tracks by their true origin:

In [3]:
is_primary = mc_example.is_track_particle(data)

def count_events(mask):
    """Number of events with at least one track whose primary MC particle passes the mask"""
    return int(ak.sum(ak.any(ak.any(mask & is_primary, axis=-1), axis=-1)))

selections = {
    "CE- (endpoint)": mc_example.is_CeMinusEndpoint(data),
    "CE- (leading log)": mc_example.is_CeMinusLeadingLog(data),
    "CE+ (leading log)": mc_example.is_CePlusLeadingLog(data),
    "DIO in target": mc_example.is_target_DIO(data),
    "Cosmic": mc_example.is_cosmic(data),
    "True electron": mc_example.is_electron(data),
}

print(f"Total events: {len(data)}")
for name, mask in selections.items():
    print(f"  {name}: {count_events(mask)}")

[pymcutil] [OK] Returning mask for is_track_particle
[pymcutil] [OK] Returning mask for start_process
[pymcutil] [OK] Returning mask for start_process
[pymcutil] [OK] Returning mask for start_process
[pymcutil] [OK] Returning mask for is_target_DIO
[pymcutil] [OK] Returning mask for is_cosmic
[pymcutil] [OK] Returning mask for is_electron
Total events: 28076
  CE- (endpoint): 0
  CE- (leading log): 106
  CE+ (leading log): 0
  DIO in target: 13085
  Cosmic: 14885
  True electron: 18398
